In [15]:
import pandas as pd

In [16]:
# ---------- 1. Load data ----------
articles = pd.read_excel("https://docs.google.com/spreadsheets/d/1DusEJ-929fOrYbs3TDOMoDV9UdmLY2PnsgYQLM1L9FM/export?format=xlsx", engine="openpyxl")
editors = pd.read_excel("https://docs.google.com/spreadsheets/d/1J6QUtxuFhugh4EX5Xe9MRrIsQ6XPYWi8Ulh1M0FZRUY/export?format=xlsx", engine="openpyxl")

# Normalize article fields
articles["Category"] = articles["Category"].str.strip().str.lower()
articles["Region"] = articles["Region"].str.strip().str.lower()

# Normalize reviewer columns
for col in ["Reviewer 1", "Reviewer 2"]:
    if col not in articles.columns:
        articles[col] = None
    articles[col] = articles[col].astype("string").str.strip()

# Normalize editor preference lists
for col in ["Topics", "Regions"]:
    editors[col] = editors[col].fillna("").apply(
        lambda s: [x.strip().lower() for x in s.split(",") if x.strip()]
    )

articles = articles[~articles["Title"].isnull()]

In [17]:
# ---------- 2. Initial load: count existing assignments ----------
current_load = dict(editors["Your Name"].astype(str).map(lambda x: (x, 0)))
current_load = {k: 0 for k in editors["Your Name"].astype(str)}

for col in ["Reviewer 1", "Reviewer 2"]:
    counts = (
        articles[col]
        .dropna()
        .astype(str)
        .value_counts()
        .to_dict()
    )
    for ed, c in counts.items():
        if ed in current_load:
            current_load[ed] += c

In [18]:
# ---------- 3. Decide max_load per editor ----------
num_articles = len(articles)
num_editors = len(editors)

if "max_load" in editors.columns:
    max_load = editors.set_index("editor")["max_load"].to_dict()
else:
    base = (num_articles * 2) // num_editors  # 2 reviewers per article
    extra = (num_articles * 2) % num_editors
    max_load = {}
    editor_names = editors["Your Name"].tolist()
    for i, name in enumerate(editor_names):
        max_load[name] = base + (1 if i < extra else 0)

# Make sure no one is already above max_load; if so, allow their current count
for ed in current_load:
    if ed not in max_load:
        max_load[ed] = current_load[ed]
    else:
        max_load[ed] = max(max_load[ed], current_load[ed])

In [19]:
# ---------- 4. Build all (article, editor) pairs and score ----------
pairs = pd.merge(
    articles[["Title", "Category", "Region"]].assign(key=1),
    editors.assign(key=1),
    on="key"
).drop(columns=["key"])

def score_row(row):
    theme_match = 1 if row["Category"] in row["Topics"] else 0
    region_match = 1 if row["Region"] in row["Regions"] else 0
    return theme_match + region_match

pairs["score"] = pairs.apply(score_row, axis=1)

In [20]:
# ---------- 5. Helper: choose next best editor for an article ----------
def choose_editor_for_article(article_id, already_assigned):
    g = pairs[pairs["Title"] == article_id].copy()
    g = g[~g["Your Name"].isin(already_assigned)]

    if g.empty:
        # fallback: anyone not already assigned
        candidates = [e for e in current_load if e not in already_assigned]
        if not candidates:
            return None
        return min(candidates, key=lambda e: current_load[e])

    g["curr_load"] = g["Your Name"].map(current_load)
    g = g.sort_values(["score", "curr_load"], ascending=[False, True])

    # First pass: positive score and capacity
    for _, row in g.iterrows():
        ed = row["Your Name"]
        if row["score"] <= 0:
            break
        if current_load[ed] < max_load[ed]:
            return ed

    # Second pass: ignore score, pick least loaded with capacity
    cap = g[g["Your Name"].map(lambda e: current_load[e] < max_load[e])]
    if not cap.empty:
        cap = cap.sort_values("curr_load", ascending=True)
        return cap.iloc[0]["Your Name"]

    # Last resort: globally least loaded, even if above nominal max
    return min(current_load, key=current_load.get)

In [21]:
# Use the column name "#" as the article identifier
ARTICLE_ID_COL = "#"

# ---------- 6. Fill missing Reviewer 1 / Reviewer 2 ----------
for idx, row in articles.iterrows():
    article_id = row[ARTICLE_ID_COL]

    # Get current values as clean strings
    r1 = str(row["Reviewer 1"]).strip() if pd.notna(row["Reviewer 1"]) else ""
    r2 = str(row["Reviewer 2"]).strip() if pd.notna(row["Reviewer 2"]) else ""

    # Track who is already on this article (we won't assign them again)
    already = set()
    if r1 != "" and r1 != "<NA>":
        already.add(r1)
    if r2 != "" and r2 != "<NA>":
        already.add(r2)

    # How many reviewers are actually assigned now?
    assigned_count = len([x for x in [r1, r2] if x not in ["", "<NA>"]])

    # If we already have 2 reviewers (even if historically same person), skip
    if assigned_count >= 2:
        continue

    # We want 2 reviewers total
    needed = 2 - assigned_count
    if needed <= 0:
        continue

    # Fill missing slots
    for _ in range(needed):
        ed = choose_editor_for_article(article_id, already)
        if ed is None:
            break

        ed = str(ed).strip()
        if ed in already:
            continue  # just in case

        already.add(ed)
        current_load[ed] += 1

        # Fill Reviewer 1 if it is empty / <NA>
        if r1 in ["", "<NA>"]:
            articles.at[idx, "Reviewer 1"] = ed
            r1 = ed
        # Else fill Reviewer 2 if it is empty / <NA> and different from Reviewer 1
        elif r2 in ["", "<NA>"] and ed != r1:
            articles.at[idx, "Reviewer 2"] = ed
            r2 = ed

In [22]:
# ---------- 7. Save updated file ----------
articles.to_excel("output/assigned_editors_multiple.xlsx", index=False)
print("Done. Saved to output/assigned_editors_multiple.xlsx")

Done. Saved to output/assigned_editors_multiple.xlsx


In [23]:
# ---------- 8. Sanity check ----------

import pandas as pd

# Ensure reviewers are strings
articles["Reviewer 1"] = articles["Reviewer 1"].astype("string").str.strip()
articles["Reviewer 2"] = articles["Reviewer 2"].astype("string").str.strip()

In [24]:
load_r1 = articles["Reviewer 1"].value_counts()
load_r2 = articles["Reviewer 2"].value_counts()
total_load = (load_r1.add(load_r2, fill_value=0)
                        .sort_values(ascending=False))

print("Total reviews per editor:")
print(total_load)

print("\nBasic stats on total load:")
print(total_load.describe())

Total reviews per editor:
Reviewer 1
Bianca Chiu                 6
Esin Nizamoglu              6
Ishita Bagri                6
Sindy Tsai-Chieh Lin        6
Arun Dayanandan             5
Ashmeet Siali               5
Camden Dyer                 5
Ben Dornyei                 5
Erzhen Wei                  5
Daphne Yao                  5
Ethan Chiu                  5
Grace Wongchaiwat           5
Hrishikesh Kayshap          5
Abhinav Veerina             5
Ignacio Sanchez Urdaneta    5
Kamryn McDonald             5
Khonzoda Murodkhujaeva      5
Katrin Lewis                5
Mercedes McCambridge        5
Michelle Jiang              5
Muhammad Umar Draaz         5
Layla Weiss                 5
Nicola McPherson            5
Nikhil Sangroula            5
Rachel Lee                  5
Nikolas Dejbord Sawan       5
Simrah Palla                5
Tarari Kalango              5
Tyson Zhang                 5
Valeriia Koryttseva         5
Name: count, dtype: Int64

Basic stats on total load:
count    

In [25]:
print("\nReviewer 1 counts:")
print(load_r1.sort_values(ascending=False))

print("\nReviewer 2 counts:")
print(load_r2.sort_values(ascending=False))


Reviewer 1 counts:
Reviewer 1
Sindy Tsai-Chieh Lin        5
Ishita Bagri                5
Ethan Chiu                  4
Layla Weiss                 4
Grace Wongchaiwat           4
Kamryn McDonald             4
Tyson Zhang                 4
Daphne Yao                  4
Camden Dyer                 4
Simrah Palla                4
Nicola McPherson            4
Nikhil Sangroula            4
Muhammad Umar Draaz         4
Arun Dayanandan             4
Tarari Kalango              3
Nikolas Dejbord Sawan       2
Esin Nizamoglu              1
Bianca Chiu                 1
Hrishikesh Kayshap          1
Erzhen Wei                  1
Michelle Jiang              1
Valeriia Koryttseva         1
Mercedes McCambridge        1
Katrin Lewis                1
Ashmeet Siali               1
Ben Dornyei                 1
Khonzoda Murodkhujaeva      1
Abhinav Veerina             1
Ignacio Sanchez Urdaneta    1
Rachel Lee                  1
Name: count, dtype: Int64

Reviewer 2 counts:
Reviewer 2
Esin Nizamog

In [26]:
dupes = articles[articles["Reviewer 1"] == articles["Reviewer 2"]]
print(f"\nArticles with the same person in both slots: {len(dupes)}")


Articles with the same person in both slots: 0


In [27]:
# Melt reviewers into one column to analyze by region/category
long = pd.melt(
    articles,
    id_vars=["#", "Category", "Region"],
    value_vars=["Reviewer 1", "Reviewer 2"],
    var_name="slot",
    value_name="editor"
).dropna(subset=["editor"])

# Reviews per editor by region
by_region = pd.crosstab(long["editor"], long["Region"])
print("\nReviews per editor by region:")
print(by_region)

# Reviews per editor by category
by_cat = pd.crosstab(long["editor"], long["Category"])
print("\nReviews per editor by category:")
print(by_cat)


Reviews per editor by region:
Region                    africa  central asia/eurasia  cross-regional/global  \
editor                                                                          
Abhinav Veerina                0                     0                      0   
Arun Dayanandan                0                     0                      1   
Ashmeet Siali                  0                     0                      4   
Ben Dornyei                    3                     0                      2   
Bianca Chiu                    2                     1                      1   
Camden Dyer                    2                     0                      3   
Daphne Yao                     0                     0                      4   
Erzhen Wei                     1                     0                      2   
Esin Nizamoglu                 0                     0                      2   
Ethan Chiu                     1                     0                      1 

In [28]:
# Melt reviewers into one column to analyze by region/category
long = pd.melt(
    articles,
    id_vars=["#", "Category", "Region"],
    value_vars=["Reviewer 1", "Reviewer 2"],
    var_name="slot",
    value_name="editor"
).dropna(subset=["editor"])

# Reviews per editor by region
by_region = pd.crosstab(long["editor"], long["Region"])
print("\nReviews per editor by region:")
print(by_region)

# Reviews per editor by category
by_cat = pd.crosstab(long["editor"], long["Category"])
print("\nReviews per editor by category:")
print(by_cat)


Reviews per editor by region:
Region                    africa  central asia/eurasia  cross-regional/global  \
editor                                                                          
Abhinav Veerina                0                     0                      0   
Arun Dayanandan                0                     0                      1   
Ashmeet Siali                  0                     0                      4   
Ben Dornyei                    3                     0                      2   
Bianca Chiu                    2                     1                      1   
Camden Dyer                    2                     0                      3   
Daphne Yao                     0                     0                      4   
Erzhen Wei                     1                     0                      2   
Esin Nizamoglu                 0                     0                      2   
Ethan Chiu                     1                     0                      1 